In [7]:
# importing libraries
import numpy as np
import pandas as pd
# Classification models 
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
# Metrics
from sklearn.metrics import precision_score, recall_score, f1_score, log_loss, accuracy_score
# Ensembling models 
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier


In [2]:
# dataset reading

X_train_path = r"E:\US_Batch_AI\ML_End-to-End\ML_project\Results\Feature_Engineering_Resulst\training_data\X_train.csv"
y_train_path = r"E:\US_Batch_AI\ML_End-to-End\ML_project\Results\Feature_Engineering_Resulst\training_data\y_train.csv"

X_test_path = r"E:\US_Batch_AI\ML_End-to-End\ML_project\Results\Feature_Engineering_Resulst\testing_data\X_test.csv"
y_test_path = r"E:\US_Batch_AI\ML_End-to-End\ML_project\Results\Feature_Engineering_Resulst\testing_data\y_test.csv"

In [3]:
X_train = pd.read_csv(X_train_path)
y_train = pd.read_csv(y_train_path)
X_test = pd.read_csv(X_test_path)
y_test = pd.read_csv(y_test_path)

In [4]:
# Add ensemble models
models = {
    'LogisticRegression': LogisticRegression(random_state=42),
    'KNeighborsClassifier': KNeighborsClassifier(),
    'SVC': SVC(probability=True, random_state=42),
    'DecisionTreeClassifier': DecisionTreeClassifier(max_depth=1, random_state=42),
    'RandomForestClassifier': RandomForestClassifier(n_estimators=100, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

In [8]:
# Loss function that returns metrics
def loss(y_true, y_pred, return_metrics=False):
    pre = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    return pre, rec, f1, acc

In [9]:
# Train and evaluate all models
def train_eval_train(models, X, y):
    results = []
    for name, model in models.items():
        model.fit(X, y)
        y_pred = model.predict(X)
        y_proba = model.predict_proba(X) if hasattr(model, "predict_proba") else None

        # Calculate log loss only if probabilities are available
        ll = log_loss(y, y_proba) if y_proba is not None else None

        pre, rec, f1, acc = loss(y, y_pred)
        results.append({
            'Model': name,
            'Precision': pre,
            'Recall': rec,
            'F1 Score': f1,
            'Accuracy': acc,
            'Log Loss': ll
        })

    # Create DataFrame
    return pd.DataFrame(results).sort_values(by='F1 Score', ascending=False)

In [10]:
results_df = train_eval_train(models, X_train, y_train)


c:\Users\HP\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:1229: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\HP\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\HP\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neighbors\_classification.py:238: DataConversionWarning: A column-vector y was passed wh

In [11]:
results_df

,Model,Precision,Recall,F1 Score,Accuracy,Log Loss
4,RandomForestClassifier,1.000000,1.000000,1.000000,1.000000,0.122785
6,XGBoost,1.000000,1.000000,1.000000,1.000000,0.052134
5,GradientBoosting,0.972477,0.688312,0.806084,0.896130,0.259273
0,LogisticRegression,0.916667,0.428571,0.584071,0.808554,0.490137
3,DecisionTreeClassifier,0.928571,0.422078,0.580357,0.808554,0.478956
1,KNeighborsClassifier,0.662651,0.357143,0.464135,0.741344,0.491823
2,SVC,1.000000,0.012987,0.025641,0.690428,0.630446


In [12]:
results_ML_with_ensembling_path = r"E:\US_Batch_AI\ML_End-to-End\ML_project\Results\ML_with_ensembling"

In [13]:
results_df.to_csv(f"{results_ML_with_ensembling_path}/results_ML_with_ensembling_path.csv", index=False)

✅ Top Performers

🥇 XGBoost

- Perfect precision, recall, F1, accuracy.

- Lowest log loss (0.052), indicating highly confident and correct probability predictions.

- Most balanced and best all-rounder.

- Might be overfitting if the dataset is small — verify with cross-validation.


🥈 Random Forest

- Also perfect scores across the board.

- Slightly higher log loss than XGBoost, but still very low (0.122).

- Also could be overfitting — again, check with test data or cross-validation.


🥉 Gradient Boosting

- Very strong, though not perfect.

- F1 score: 0.81 – better generalization than Logistic Regression or Decision Trees.

- Lower log loss means more calibrated predictions